# Chapter 2 — Vanilla RAG End-to-End (Groq + HuggingFace)

In [3]:
import os, getpass

# Embeddings are LOCAL (HF) — no key needed.
# Generation is Groq — needs a key. Read it from env or prompt once.
if not os.environ.get("GROQ_API_KEY"):
    os.environ["GROQ_API_KEY"] = getpass.getpass("Paste your GROQ_API_KEY: ")

print("Groq key set:", bool(os.environ.get("GROQ_API_KEY")))

Groq key set: True


## Embeddings

**The one trick that makes retrieval a single matrix multiply.** Cosine similarity is
$$\cos(a,b)=\frac{a\cdot b}{\lVert a\rVert\,\lVert b\rVert}.$$
If we **L2-normalize** every vector to unit length at index time, then $\lVert a\rVert=\lVert b\rVert=1$
and cosine collapses to a plain dot product $a\cdot b$. So the whole search step becomes
`matrix @ query`. Normalize **once**, never again.

`sentence-transformers` does this for us with `normalize_embeddings=True`.

In [4]:
import numpy as np
from sentence_transformers import SentenceTransformer

MODEL = "BAAI/bge-small-en-v1.5"     # 384-dim, strong & fast, runs locally, free
_model = SentenceTransformer(MODEL)  # first call downloads the weights (~130 MB)

# bge is ASYMMETRIC: instruction on the QUERY only, never on passages.
QUERY_PREFIX = "Represent this sentence for searching relevant passages: "

def embed(texts, batch_size=128, is_query=False):
    """Return an L2-normalized (n, d) float32 matrix. Local inference, no network."""
    texts = [t.replace("\n", " ") for t in texts]     # newlines still hurt some models
    if is_query:
        texts = [QUERY_PREFIX + t for t in texts]
    m = _model.encode(
        texts,
        batch_size=batch_size,
        normalize_embeddings=True,   # unit vectors -> dot == cosine
        convert_to_numpy=True,
        show_progress_bar=False,
    )
    return m.astype(np.float32)

# Sanity checks: right shape, and vectors really are unit length.
v = embed(["hello world", "a second sentence"])
print("shape:", v.shape)                                   
print("norms:", np.linalg.norm(v, axis=1))                 
assert np.allclose(np.linalg.norm(v, axis=1), 1.0, atol=1e-4)

e:\Learning\Preparation\Retrieval-Augmented-Generation\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 3350.22it/s]


shape: (2, 384)
norms: [1. 1.]


## The store — brute force done right

**Why brute force before FAISS?** At under ~100k chunks, an exact NumPy scan is *simpler and
faster* than any approximate-nearest-neighbor (ANN) index, and — crucially — it gives a
**ground-truth baseline**. When we add ANN later, we measure its recall *against this*.

**One line worth knowing:** `argsort(-sims)[:k]` is `O(N log N)`. For large `N`, use
`np.argpartition(-sims, k)[:k]` (`O(N)`) and sort only the `k` survivors — roughly 10× faster
at `N=1M`.

In [5]:
import pickle

class VectorStore:
    def __init__(self):
        self.matrix = None                 # (N, d) float32, unit vectors
        self.meta = []                     # list of dicts: {text, source, chunk_id, ...}

    def add(self, vectors, metadatas):
        self.matrix = vectors if self.matrix is None else np.vstack([self.matrix, vectors])
        self.meta.extend(metadatas)

    def search(self, qvec, k=5, where=None):
        """qvec: (d,) unit vector. where: dict metadata filter (pre-filter)."""
        idx = np.arange(len(self.meta))
        if where:                                          # metadata pre-filter (Ch 5, Ch 20)
            idx = np.array([i for i in idx
                            if all(self.meta[i].get(kk) == vv for kk, vv in where.items())])
            if len(idx) == 0:
                return []
        sims = self.matrix[idx] @ qvec                     # cosine scores (unit vectors)
        top = idx[np.argsort(-sims)[:k]]
        return [{**self.meta[i], "score": float(self.matrix[i] @ qvec)} for i in top]

    def save(self, path):
        with open(path, "wb") as f: pickle.dump((self.matrix, self.meta), f)
    def load(self, path):
        with open(path, "rb") as f: self.matrix, self.meta = pickle.load(f)

print("VectorStore ready.")

VectorStore ready.


## A tiny built-in corpus

In [6]:
import os, json, pathlib
pathlib.Path("corpus").mkdir(exist_ok=True)

DOCS = {
    "corpus/policy.txt": (
        "Returns and refunds. Customers may return any item within 30 days of "
        "delivery for a full refund. The refund window is 30 days. Items must be "
        "unused and in original packaging. Shipping costs are non-refundable. "
        "Refunds are processed to the original payment method within 5 business days."
    ),
    "corpus/about.txt": (
        "About Acme Corp. Acme Corp was founded in 2019 and is headquartered in "
        "Austin, Texas. The Chief Executive Officer is Jane Doe. The company builds "
        "developer tools for data teams and employs about 120 people."
    ),
    "corpus/support.txt": (
        "Support hours. Customer support is available Monday through Friday, 9am to "
        "6pm Central Time. Support can be reached by email at help@acme.example. "
        "Enterprise customers get a dedicated Slack channel and a 4-hour response SLA."
    ),
}
for path, text in DOCS.items():
    with open(path, "w", encoding="utf-8") as f:
        f.write(text)

GOLDEN = [
    {"q": "What is the refund window?", "answer": "30 days",
     "relevant_ids": ["corpus/policy.txt#0"]},
    {"q": "Who is the CEO?", "answer": "Jane Doe",
     "relevant_ids": ["corpus/about.txt#0"]},
    {"q": "When is support available?", "answer": "Mon-Fri 9am-6pm Central",
     "relevant_ids": ["corpus/support.txt#0"]},
]
with open("golden.jsonl", "w", encoding="utf-8") as f:
    for row in GOLDEN:
        f.write(json.dumps(row) + "\n")

print("Wrote", len(DOCS), "docs and", len(GOLDEN), "golden items.")

Wrote 3 docs and 3 golden items.


## Chunking, Retrieval, Generation

### Token-accurate chunking
Word-splitting **undercounts** versus the tokens a model actually sees, so word-based chunks
silently blow past the embedder's limit and get **truncated** (the tail is lost). We chunk on
`tiktoken` tokens with overlap so adjacent chunks share context.

> `bge-small` has a **512-token** input limit, so `max_tokens=400` stays safely under it — no
> truncation. (Purists: use the embedding model's *own* tokenizer for perfect accuracy;

In [7]:
import glob, tiktoken
from openai import OpenAI

enc = tiktoken.get_encoding("cl100k_base")

client = OpenAI(
    base_url="https://api.groq.com/openai/v1",
    api_key=os.environ["GROQ_API_KEY"],
)
GEN_MODEL = "openai/gpt-oss-20b"


In [8]:
def chunk(text, max_tokens=400, overlap=60):
    toks = enc.encode(text)
    step = max_tokens - overlap
    out = []
    for i in range(0, len(toks), step):
        out.append(enc.decode(toks[i:i + max_tokens]))
        if i + max_tokens >= len(toks):
            break
    return out

In [9]:
def build_index(folder="corpus", store_path="index.pkl"):
    store = VectorStore()
    texts, metas = [], []
    for path in glob.glob(f"{folder}/**/*.txt", recursive=True):
        doc = open(path, encoding="utf-8").read()
        for ci, c in enumerate(chunk(doc)):
            texts.append(c)
            metas.append({"text": c, "source": path, "chunk_id": f"{path}#{ci}"})
    store.add(embed(texts), metas)                 # passages: is_query=False (default)
    store.save(store_path)
    print(f"indexed {len(texts)} chunks from {folder}/")
    return store

store = build_index()

indexed 3 chunks from corpus/


## The prompt contract

Three clauses, each killing a specific failure mode:

| Clause | Strong version | Failure it kills |
|---|---|---|
| **Grounding** | "Answer *strictly* from the context. Do not use outside knowledge." | Answering from stale parametric memory |
| **Abstention** | "If not in context, reply *exactly*: I don't know." | Confident hallucination on out-of-corpus queries |
| **Citation** | "Cite inline as [1], [2]." + numbered blocks | Unattributable output |

We also order context **best-chunk-first** to fight *lost-in-the-middle* — models attend
worst to the middle of a long context, so the top hit should never be buried.

In [10]:
SYSTEM = (
    "You answer strictly from the provided context. "
    "If the answer is not in the context, reply exactly: I don't know. "
    "Cite the sources you used inline as [1], [2]. Do not use outside knowledge."
)

def build_prompt(query, chunks):
    # best-first ordering fights lost-in-the-middle; our search already sorts by score desc
    block = "\n\n".join(f"[{i+1}] (source: {c['source']})\n{c['text']}"
                        for i, c in enumerate(chunks))
    return f"Context:\n{block}\n\nQuestion: {query}\nAnswer:"

def answer(query, store, k=5, model=GEN_MODEL, system=SYSTEM):
    qvec = embed([query], is_query=True)[0]        # QUERY side -> is_query=True
    hits = store.search(qvec, k=k)
    resp = client.chat.completions.create(
        model=model, temperature=0,                # temperature=0 -> reproducible RAG
        messages=[{"role": "system", "content": system},
                  {"role": "user", "content": build_prompt(query, hits)}])
    return {"answer": resp.choices[0].message.content,
            "contexts": hits,
            "citations": [h["source"] for h in hits]}

In [11]:
r = answer("What is the refund window?", store, k=3)

print(r["answer"])
print("---- retrieved ----")
for c in r["contexts"]:
    print(f"  {c['score']:.3f}  {c['chunk_id']}")

The refund window is 30 days. [1]
---- retrieved ----
  0.712  corpus\policy.txt#0
  0.468  corpus\support.txt#0
  0.351  corpus\about.txt#0


## The abstention clause, measured

The cheapest hallucination fix in RAG is one sentence in the system prompt. Let's **prove** it
with an A/B on an *out-of-corpus* question — one whose answer isn't in our tiny corpus. A good
system should refuse ("I don't know") rather than fabricate.

In [12]:
WEAK_SYSTEM = "Answer the question using the context if helpful."   # no abstention clause

off_corpus = [
    "What is the company's stock price?",
    "What is the capital of France?",
    "Who is the VP of Engineering?",
]

def refuses(text):
    return text.strip().lower().startswith("i don't know")

for sys_name, sys_prompt in [("WEAK (no abstention)", WEAK_SYSTEM),
                             ("STRONG (abstention)", SYSTEM)]:
    refused = sum(refuses(answer(q, store, k=3, system=sys_prompt)["answer"])
                  for q in off_corpus)
    print(f"{sys_name:24s}  abstention rate: {refused}/{len(off_corpus)}")

WEAK (no abstention)      abstention rate: 0/3
STRONG (abstention)       abstention rate: 3/3


## The eval harness

A RAG with no eval is unfalsifiable. Two families of metric:

- **Retrieval** (no LLM needed, pure math): **recall@k** — did the relevant chunk make the
  top-k? — and **MRR** (mean reciprocal rank) — how *high* did it rank?
- **Generation**: **faithfulness** via LLM-as-judge — is every claim in the answer supported by
  the retrieved context?

`recall@k` / `MRR` are provider-agnostic. Only the faithfulness judge calls Groq.

In [13]:
GOLDEN = [json.loads(l) for l in open("golden.jsonl", encoding="utf-8")]

def recall_at_k(store, k=5):
    hits, rr = [], []
    for item in GOLDEN:
        qvec = embed([item["q"]], is_query=True)[0]
        got = [h["chunk_id"] for h in store.search(qvec, k=k)]
        rel = set(item["relevant_ids"])
        hits.append(len(rel & set(got)) / len(rel))                    # recall@k
        rank = next((i + 1 for i, g in enumerate(got) if g in rel), None)
        rr.append(1 / rank if rank else 0.0)                           # reciprocal rank
    return {"recall@k": float(np.mean(hits)), "MRR": float(np.mean(rr))}

print(recall_at_k(store, k=3))

{'recall@k': 0.0, 'MRR': 0.0}


In [14]:
FAITH_PROMPT = """You are grading faithfulness. Given CONTEXT and ANSWER, is every
factual claim in ANSWER supported by CONTEXT? Reply only YES or NO.
CONTEXT:
{ctx}

ANSWER:
{ans}"""

def faithfulness(store, k=5, judge_model=GEN_MODEL):
    good = 0
    for item in GOLDEN:
        r = answer(item["q"], store, k=k)
        ctx = "\n".join(c["text"] for c in r["contexts"])
        verdict = client.chat.completions.create(
            model=judge_model, temperature=0,
            messages=[{"role": "user",
                       "content": FAITH_PROMPT.format(ctx=ctx, ans=r["answer"])}]
        ).choices[0].message.content.strip().upper()
        good += verdict.startswith("YES")
    return good / len(GOLDEN)

print("faithfulness:", faithfulness(store, k=3))

faithfulness: 1.0


### The scorecard
This little table is the spine of the whole curriculum: every later technique (better chunking,
hybrid retrieval, rerankers, ...) has to move one of these numbers up, or it doesn't ship.

In [15]:
scores = recall_at_k(store, k=3)
scores["faithfulness"] = faithfulness(store, k=3)
print("=" * 34)
print(f"{'recall@3':<14}{scores['recall@k']:.3f}")
print(f"{'MRR':<14}{scores['MRR']:.3f}")
print(f"{'faithfulness':<14}{scores['faithfulness']:.3f}")
print("=" * 34)

recall@3      0.000
MRR           0.000
faithfulness  1.000


Per query, vanilla RAG is:

```
latency  ≈  t_embed(query)  +  t_search      +  t_generate
         ≈   ~10-30 ms (local) + <5 ms (brute, 100k) + 100-800 ms (Groq is fast)
```